# 02 · Unity Catalog: quién puede ver qué

⏱ **12 min** · Módulo 4 · Slides de la Intro: 17 (acceso) y 18 (los permisos se heredan hacia abajo)

**Objetivo:** dar, ver y quitar permisos, comprobar que se heredan, y ocultar columnas y filas sensibles.

| Paso | Qué haces |
|---|---|
| 1 | ¿Quién soy y a qué grupos pertenezco? |
| 2 | Dar un permiso sobre una tabla (`GRANT`) |
| 3 | Dar un permiso sobre el esquema: se hereda |
| 4 | Quitar permisos (`REVOKE`) |
| 5 | Ocultar una columna: máscara |
| 6 | Ocultar filas: filtro |
| 7 | Dejar la tabla como estaba |

**Requiere:** notebook 01.

In [0]:
%run ./00_Setup

## Paso 1 · ¿Quién soy y a qué grupos pertenezco?

In [0]:
%sql
SELECT current_user() AS usuario,
       is_account_group_member('account users') AS en_account_users,
       is_account_group_member('finanzas')      AS en_finanzas

👀 **Qué observar:** los permisos se dan a **grupos**, no a personas. `account users` es el grupo con todos los usuarios de la cuenta
(en la Free Edition, solo tú). El grupo `finanzas` no existe aquí; en la empresa tendrías grupos como `analistas`, `ingenieria` o `finanzas`.

## Paso 2 · Dar un permiso sobre una tabla

In [0]:
%sql
GRANT SELECT ON TABLE suscriptores TO `account users`

In [0]:
%sql
SHOW GRANTS ON TABLE suscriptores

👀 **Qué observar:** `SELECT` sobre la tabla no alcanza. Para leerla hace falta la cadena completa:
`USE CATALOG` en el catálogo + `USE SCHEMA` en el esquema + `SELECT` en la tabla.

In [0]:
%sql
GRANT USE SCHEMA ON SCHEMA lab_tvmaze_m56 TO `account users`

## Paso 3 · Dar un permiso sobre el esquema: se hereda

Un permiso dado en un nivel baja a todo lo que está dentro.

In [0]:
%sql
GRANT SELECT ON SCHEMA lab_tvmaze_m56 TO `account users`

In [0]:
%sql
-- ¿De dónde viene cada permiso? La columna inherited_from lo dice
SELECT table_name, grantee, privilege_type, inherited_from
FROM information_schema.table_privileges
WHERE table_schema = 'lab_tvmaze_m56' AND table_name LIKE 'suscriptores%' AND grantee = 'account users'
ORDER BY table_name, inherited_from

👀 **Qué observar:** `suscriptores` tiene `SELECT` dos veces: el directo (`NONE`) y el heredado (`SCHEMA`).
`suscriptores_por_pais` lo tiene sin que nadie se lo diera. Cualquier tabla que nazca en el esquema lo tendrá también.

## Paso 4 · Quitar permisos

In [0]:
%sql
REVOKE SELECT ON SCHEMA lab_tvmaze_m56 FROM `account users`

In [0]:
%sql
REVOKE SELECT ON TABLE suscriptores FROM `account users`

In [0]:
%sql
SHOW GRANTS ON SCHEMA lab_tvmaze_m56

👀 **Qué observar:** solo queda `USE SCHEMA`: el grupo puede entrar al esquema, pero no leer ninguna tabla.
Es el caso del slide 17: *"Ventas ve la tabla de ventas, pero no la de salarios"*.

## Paso 5 · Ocultar una columna: máscara

A veces alguien debe ver la tabla, pero no un dato sensible. Una **máscara** es una función que decide qué se muestra en una columna.

In [0]:
%sql
CREATE OR REPLACE FUNCTION ocultar_email(email STRING)
RETURN CASE WHEN is_account_group_member('finanzas') THEN email ELSE '***' END

In [0]:
%sql
ALTER TABLE suscriptores ALTER COLUMN email SET MASK ocultar_email

In [0]:
%sql
SELECT * FROM suscriptores

👀 **Qué observar:** el email sale como `***` incluso para ti, que eres el dueño, porque no estás en `finanzas`.
La tabla guarda el email real: la máscara se aplica al leer.

## Paso 6 · Ocultar filas: filtro

Un **filtro de filas** es una función que decide qué filas ve cada quien.

In [0]:
%sql
CREATE OR REPLACE FUNCTION solo_peru(pais STRING)
RETURN is_account_group_member('gerencia') OR pais = 'Perú'

In [0]:
%sql
ALTER TABLE suscriptores SET ROW FILTER solo_peru ON (pais)

In [0]:
%sql
SELECT * FROM suscriptores

In [0]:
%sql
SELECT * FROM suscriptores_por_pais

👀 **Qué observar:** solo ves los suscriptores de Perú, y la vista tampoco muestra los demás: el filtro vive en la tabla,
así que no se puede esquivar consultando por otro lado.

## Paso 7 · Dejar la tabla como estaba

In [0]:
%sql
ALTER TABLE suscriptores DROP ROW FILTER

In [0]:
%sql
ALTER TABLE suscriptores ALTER COLUMN email DROP MASK

In [0]:
%sql
SELECT * FROM suscriptores

## Tres formas de limitar lo que se ve

| Necesidad | Herramienta | Ejemplo |
|---|---|---|
| Que un grupo no vea la tabla | No darle `SELECT` | Ventas no ve salarios |
| Que vea la tabla, pero no una columna | Máscara de columna | El email sale como `***` |
| Que vea solo sus filas | Filtro de filas | Cada país ve sus suscriptores |

## ✔ Checkpoint

- [ ] Sé la cadena de permisos para leer una tabla: `USE CATALOG` + `USE SCHEMA` + `SELECT`
- [ ] Vi un permiso heredado del esquema en `information_schema.table_privileges`
- [ ] Di y quité permisos con `GRANT` y `REVOKE`
- [ ] Oculté una columna con una máscara y filas con un filtro

**Siguiente:** `03_UC_Almacenamiento_Auditoria_Linaje`